# LLM Provider Comparison

## Learning Objectives

By completing this notebook, you will:
1. Compare different LLM providers (Anthropic, OpenAI, etc.) through LLM Mesh
2. Understand performance tradeoffs: quality vs speed vs cost
3. Select appropriate models for different use cases
4. Implement multi-provider failover strategies
5. Benchmark and evaluate model outputs

## Prerequisites

- Completed Notebook 01: First Connection
- Multiple LLM connections configured in Dataiku
- Understanding of token pricing models

## Estimated Time: 45 minutes

In [ ]:
learning_objectives(["Compare different LLM providers (Anthropic, OpenAI, etc.) through LLM Mesh", "Understand performance tradeoffs: quality vs speed vs cost", "Select appropriate models for different use cases", "Implement multi-provider failover strategies", "Benchmark and evaluate model outputs"])

## Why Compare Providers?

**The Problem**: Not all LLMs are equal. Each provider/model excels at different tasks:

- **Claude Opus**: Best reasoning, highest quality, most expensive
- **Claude Sonnet**: Balanced performance and cost
- **Claude Haiku**: Fastest, cheapest, good for simple tasks
- **GPT-4**: Strong general capabilities, widely adopted
- **GPT-3.5**: Very fast and cheap, suitable for simple tasks

**The Solution**: LLM Mesh lets you test providers easily and switch without code changes.

### Key Insight

The best model for production is often NOT the most capable. It's the one that:
1. Meets quality requirements
2. Stays within budget
3. Meets latency targets
4. Has acceptable reliability

## Setup and Configuration

Import libraries and configure available connections.

In [ ]:
import sys; sys.path.insert(0, '../../../../..')
from resources.notebook_style import apply_course_theme, learning_objectives, section_divider, callout, key_takeaways
from resources.graphics.plot_theme import apply_plot_theme

In [ ]:
# Apply course styling
apply_course_theme()
apply_plot_theme()

In [ ]:
import dataiku
# Note: RateLimitError may not be importable from dataiku.llm
# Handle rate limit errors with try/except on generic exceptions
import dataiku  # Use project.get_llm() for LLM access
import json
import time
from typing import Dict, List
import pandas as pd
import matplotlib.pyplot as plt
from statistics import mean, stdev

## Discover Available Connections

Let's find all configured LLM connections and their properties.

In [ ]:
# Configuration: Update with your connections
CONNECTIONS = {
    'claude-opus': {'model': 'claude-opus-4', 'tier': 'premium'},
    'claude-sonnet': {'model': 'claude-sonnet-4', 'tier': 'standard'},
    'claude-haiku': {'model': 'claude-haiku', 'tier': 'economy'},
    'gpt4': {'model': 'gpt-4o', 'tier': 'standard'},
    'gpt35': {'model': 'gpt-3.5-turbo', 'tier': 'economy'}
}

# Pricing (USD per 1M tokens)
PRICING = {
    'claude-opus-4': {'input': 15.0, 'output': 75.0},
    'claude-sonnet-4': {'input': 3.0, 'output': 15.0},
    'claude-haiku': {'input': 0.25, 'output': 1.25},
    'gpt-4o': {'input': 2.5, 'output': 10.0},
    'gpt-3.5-turbo': {'input': 0.5, 'output': 1.5}
}

print("Configured connections:")
for name, config in CONNECTIONS.items():
    model = config['model']
    pricing = PRICING.get(model, {'input': 0, 'output': 0})
    print(f"  {name}: {model} (${pricing['input']:.2f}/${pricing['output']:.2f} per 1M tokens)")

## Exercise 1: Basic Provider Comparison

Compare how different providers respond to the same prompt. Notice differences in:
- Response quality
- Response length
- Latency
- Token usage

In [ ]:
def compare_providers(prompt: str, connections: List[str], max_tokens: int = 200) -> pd.DataFrame:
    """
    Send same prompt to multiple providers and compare results.
    
    Parameters
    ----------
    prompt : str
        The prompt to send
    connections : List[str]
        List of connection names to test
    max_tokens : int
        Maximum tokens for response
        
    Returns
    -------
    pd.DataFrame
        Comparison results with metrics
    """
    results = []
    
    for conn_name in connections:
        try:
            llm = LLM(conn_name)
            
            # Measure latency
            start_time = time.time()
            response = llm.complete(prompt, max_tokens=max_tokens, temperature=0.3)
            latency = time.time() - start_time
            
            # Calculate cost
            model = CONNECTIONS[conn_name]['model']
            pricing = PRICING.get(model, {'input': 0, 'output': 0})
            cost = (
                response.usage.prompt_tokens * pricing['input'] / 1_000_000 +
                response.usage.completion_tokens * pricing['output'] / 1_000_000
            )
            
            results.append({
                'connection': conn_name,
                'model': model,
                'response': response.text,
                'latency_sec': round(latency, 2),
                'total_tokens': response.usage.total_tokens,
                'cost_usd': round(cost, 6),
                'response_length': len(response.text)
            })
            
        except Exception as e:
            results.append({
                'connection': conn_name,
                'model': CONNECTIONS.get(conn_name, {}).get('model', 'unknown'),
                'response': f"ERROR: {str(e)}",
                'latency_sec': None,
                'total_tokens': 0,
                'cost_usd': 0,
                'response_length': 0
            })
    
    return pd.DataFrame(results)

# Test with a commodity analysis question
test_prompt = """Explain the concept of 'crack spread' in oil refining and why it matters to commodity traders. Be concise."""

comparison = compare_providers(
    test_prompt,
    connections=['claude-sonnet', 'claude-haiku', 'gpt35'],  # Update with your connections
    max_tokens=200
)

# Display comparison
print("\n=== PROVIDER COMPARISON ===")
print(comparison[['connection', 'latency_sec', 'total_tokens', 'cost_usd']].to_string(index=False))
print("\n=== RESPONSES ===")
for _, row in comparison.iterrows():
    print(f"\n{row['connection'].upper()}:")
    print(row['response'][:300] + "..." if len(row['response']) > 300 else row['response'])

### Exercise 1.1: Quality vs Cost Analysis

**Task**: Analyze the tradeoff between response quality and cost. Which provider gives the best value?

In [ ]:
# TODO: Calculate quality score (your own assessment or use LLM-as-judge)
# TODO: Plot cost vs quality

def assess_quality(response: str, criteria: str) -> int:
    """
    Assess response quality on 1-5 scale.
    
    For now, use simple heuristics:
    - Mentions key terms (crack spread, refining margin, etc.)
    - Explains the concept
    - Mentions why it matters
    
    Returns score 1-5.
    """
    score = 3  # Start with baseline
    
    # YOUR CODE HERE
    # Check for key terms: crack spread, refining margin, crude, products
    # Check for explanation structure
    # Check for trader relevance
    
    return score

# Apply quality assessment
comparison['quality_score'] = comparison['response'].apply(
    lambda r: assess_quality(r, "crack spread explanation")
)

# Calculate value score (quality per dollar)
comparison['value_score'] = comparison.apply(
    lambda row: (row['quality_score'] / (row['cost_usd'] * 1000)) if row['cost_usd'] > 0 else 0,
    axis=1
)

print("\nQuality vs Cost Analysis:")
print(comparison[['connection', 'quality_score', 'cost_usd', 'value_score']].to_string(index=False))

# Visualization
fig, ax = plt.subplots(figsize=(10, 6))
ax.scatter(comparison['cost_usd'] * 1000, comparison['quality_score'], s=200, alpha=0.6)

for _, row in comparison.iterrows():
    ax.annotate(row['connection'], 
                (row['cost_usd'] * 1000, row['quality_score']),
                fontsize=9, ha='center')

ax.set_xlabel('Cost (cents)', fontsize=12)
ax.set_ylabel('Quality Score (1-5)', fontsize=12)
ax.set_title('LLM Provider: Quality vs Cost', fontsize=14)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Auto-graded test
assert 'quality_score' in comparison.columns, "Missing quality_score column"
assert all(1 <= s <= 5 for s in comparison['quality_score']), "Quality scores must be 1-5"
print("\n✓ Exercise 1.1 passed!")

## Exercise 2: Task-Specific Benchmarking

Different models excel at different tasks. Let's benchmark on multiple task types:
1. **Structured extraction**: Parsing reports into JSON
2. **Classification**: Sentiment analysis
3. **Reasoning**: Complex analytical questions
4. **Summarization**: Condensing long text

In [ ]:
# Benchmark dataset
BENCHMARK_TASKS = {
    'extraction': {
        'prompt': '''Extract key data from this report as JSON with fields: commodity, price, change_pct, volume.
        
Report: WTI crude settled at $79.45, up 3.2% on volume of 425,000 contracts.

JSON:''',
        'expected': {'commodity': 'WTI crude', 'price': 79.45, 'change_pct': 3.2},
        'max_tokens': 100
    },
    'classification': {
        'prompt': '''Classify sentiment as BULLISH, BEARISH, or NEUTRAL:
        
"OPEC+ surprised markets with a 500k bpd production cut, tightening an already constrained market."

Sentiment:''',
        'expected': 'BULLISH',
        'max_tokens': 10
    },
    'reasoning': {
        'prompt': '''If crude oil inventories fall by 5M barrels but refinery utilization also drops 2%, what does this suggest about demand vs supply dynamics? Answer in 2 sentences.''',
        'expected_keywords': ['demand', 'supply', 'inventory', 'draw'],
        'max_tokens': 150
    },
    'summarization': {
        'prompt': '''Summarize in one sentence:
        
The Energy Information Administration reported that U.S. crude oil inventories decreased by 7.2 million barrels during the week ending March 15. This was significantly larger than the analyst consensus of a 2.1 million barrel draw. The larger-than-expected decline was driven by a combination of strong refinery runs at 92% utilization and reduced imports from Canada due to pipeline maintenance. At 445.3 million barrels, inventories are now 2% below the five-year average.

Summary:''',
        'expected_keywords': ['7.2 million', 'draw', 'inventories'],
        'max_tokens': 50
    }
}

def benchmark_task(task_name: str, task_config: dict, connections: List[str]) -> pd.DataFrame:
    """
    Benchmark providers on a specific task type.
    """
    results = []
    
    for conn_name in connections:
        try:
            llm = LLM(conn_name)
            
            start_time = time.time()
            response = llm.complete(
                task_config['prompt'],
                max_tokens=task_config['max_tokens'],
                temperature=0  # Deterministic for benchmarking
            )
            latency = time.time() - start_time
            
            # Calculate cost
            model = CONNECTIONS[conn_name]['model']
            pricing = PRICING.get(model, {'input': 0, 'output': 0})
            cost = (
                response.usage.prompt_tokens * pricing['input'] / 1_000_000 +
                response.usage.completion_tokens * pricing['output'] / 1_000_000
            )
            
            results.append({
                'task': task_name,
                'connection': conn_name,
                'response': response.text.strip(),
                'latency_sec': round(latency, 2),
                'cost_usd': round(cost, 6),
                'tokens': response.usage.total_tokens
            })
            
        except Exception as e:
            results.append({
                'task': task_name,
                'connection': conn_name,
                'response': f"ERROR: {e}",
                'latency_sec': None,
                'cost_usd': 0,
                'tokens': 0
            })
    
    return pd.DataFrame(results)

# Run benchmarks
all_results = []
test_connections = ['claude-sonnet', 'claude-haiku']  # Update with your connections

for task_name, task_config in BENCHMARK_TASKS.items():
    print(f"\nBenchmarking: {task_name}...")
    task_results = benchmark_task(task_name, task_config, test_connections)
    all_results.append(task_results)
    print(task_results[['connection', 'latency_sec', 'cost_usd']].to_string(index=False))

# Combine results
benchmark_df = pd.concat(all_results, ignore_index=True)
benchmark_df.head()

### Exercise 2.1: Identify Best Model Per Task

**Task**: Analyze which model performs best for each task type, considering both quality and cost.

In [ ]:
# TODO: Score each response
# TODO: Calculate best model per task
# TODO: Create recommendation matrix

def score_response(row: pd.Series, task_config: dict) -> int:
    """
    Score response quality for a task (1-5).
    
    Your implementation should check:
    - For extraction: valid JSON with expected fields
    - For classification: matches expected value
    - For reasoning: contains expected keywords
    - For summarization: concise and contains key facts
    """
    response = row['response']
    task = row['task']
    
    # YOUR CODE HERE
    score = 3  # Baseline
    
    if task == 'extraction':
        # Check if valid JSON
        try:
            data = json.loads(response)
            # Check expected fields
            if all(k in data for k in task_config['expected'].keys()):
                score = 5
        except:
            score = 1
    
    elif task == 'classification':
        # Check if matches expected
        if task_config['expected'].lower() in response.lower():
            score = 5
        else:
            score = 2
    
    # Add logic for reasoning and summarization tasks
    
    return score

# Apply scoring
for task_name, task_config in BENCHMARK_TASKS.items():
    mask = benchmark_df['task'] == task_name
    benchmark_df.loc[mask, 'quality_score'] = benchmark_df[mask].apply(
        lambda row: score_response(row, task_config),
        axis=1
    )

# Summary by task and connection
summary = benchmark_df.groupby(['task', 'connection']).agg({
    'quality_score': 'mean',
    'latency_sec': 'mean',
    'cost_usd': 'sum'
}).round(3)

print("\nBenchmark Summary:")
print(summary)

# Best model per task
print("\n=== RECOMMENDATIONS ===")
for task in benchmark_df['task'].unique():
    task_data = benchmark_df[benchmark_df['task'] == task]
    best = task_data.loc[task_data['quality_score'].idxmax()]
    print(f"\n{task.upper()}:")
    print(f"  Best: {best['connection']} (quality: {best['quality_score']}, cost: ${best['cost_usd']:.6f})")

# Auto-graded test
assert 'quality_score' in benchmark_df.columns, "Missing quality_score"
assert not benchmark_df['quality_score'].isna().any(), "All tasks should be scored"
print("\n✓ Exercise 2.1 passed!")

## Exercise 3: Multi-Provider Failover

Production systems need resilience. If the primary provider fails or is rate-limited, automatically fail over to a backup.

### Failover Strategy

```
1. Try primary (best quality)
2. On failure, try secondary (good quality, better availability)
3. On failure, try tertiary (basic quality, very reliable)
4. If all fail, return error
```

In [ ]:
class ResilientLLM:
    """
    LLM client with automatic failover.
    
    Tries connections in priority order until one succeeds.
    """
    
    def __init__(self, connections: List[tuple]):
        """
        Parameters
        ----------
        connections : List[tuple]
            List of (name, priority) tuples, lower priority = tried first
        """
        self.connections = sorted(connections, key=lambda x: x[1])
        self.stats = {'attempts': 0, 'successes': 0, 'failovers': 0}
    
    def complete(self, prompt: str, **kwargs) -> dict:
        """
        Try completion with failover.
        
        Returns
        -------
        dict with keys: text, connection_used, attempt_number, success
        """
        last_error = None
        
        for attempt, (conn_name, priority) in enumerate(self.connections, 1):
            self.stats['attempts'] += 1
            
            try:
                llm = LLM(conn_name)
                response = llm.complete(prompt, **kwargs)
                
                # Success
                self.stats['successes'] += 1
                if attempt > 1:
                    self.stats['failovers'] += 1
                    print(f"⚠️  Failed over to {conn_name} (priority {priority})")
                
                return {
                    'text': response.text,
                    'connection_used': conn_name,
                    'attempt_number': attempt,
                    'success': True,
                    'usage': response.usage
                }
                
            except Exception as e:  # RateLimitError — verify import
                print(f"⚠️  {conn_name} rate limited, trying next...")
                last_error = e
                continue
                
            except Exception as e:
                print(f"⚠️  {conn_name} failed: {e}")
                last_error = e
                time.sleep(0.5)  # Brief delay
                continue
        
        # All failed
        raise Exception(f"All providers failed. Last error: {last_error}")
    
    def get_stats(self) -> dict:
        """Return usage statistics."""
        return {
            **self.stats,
            'success_rate': self.stats['successes'] / self.stats['attempts'] if self.stats['attempts'] > 0 else 0,
            'failover_rate': self.stats['failovers'] / self.stats['successes'] if self.stats['successes'] > 0 else 0
        }

# Test failover
resilient_llm = ResilientLLM([
    ('claude-sonnet', 1),   # Primary
    ('claude-haiku', 2),    # Secondary
    ('gpt35', 3)            # Tertiary
])

# Make some requests
test_prompts = [
    "What is OPEC+?",
    "Explain contango.",
    "What affects oil prices?"
]

for prompt in test_prompts:
    result = resilient_llm.complete(prompt, max_tokens=100)
    print(f"✓ Success via {result['connection_used']} (attempt {result['attempt_number']})")

# Show statistics
print("\nFailover Statistics:")
stats = resilient_llm.get_stats()
for key, value in stats.items():
    print(f"  {key}: {value}")

### Exercise 3.1: Intelligent Routing

**Task**: Extend the `ResilientLLM` class to route based on task complexity:
- Simple tasks (< 50 tokens) → Use cheapest model
- Medium tasks (50-200 tokens) → Use balanced model
- Complex tasks (> 200 tokens) → Use best model

In [ ]:
class IntelligentRouter(ResilientLLM):
    """
    Routes to appropriate model based on task complexity.
    """
    
    def __init__(self):
        # Define tiers
        self.tiers = {
            'economy': [('claude-haiku', 1), ('gpt35', 2)],
            'standard': [('claude-sonnet', 1), ('gpt4', 2)],
            'premium': [('claude-opus', 1), ('gpt4', 2)]
        }
        self.stats = {'attempts': 0, 'successes': 0, 'failovers': 0, 'routing': {}}
    
    def estimate_complexity(self, prompt: str) -> str:
        """
        Estimate task complexity from prompt.
        
        Returns 'economy', 'standard', or 'premium'
        """
        # YOUR CODE HERE
        # Estimate based on:
        # - Prompt length
        # - Keywords ("analyze", "compare", "explain" vs "what is")
        # - Expected output complexity
        
        prompt_len = len(prompt.split())
        
        if prompt_len < 20:
            return 'economy'
        elif prompt_len < 100:
            return 'standard'
        else:
            return 'premium'
    
    def complete(self, prompt: str, **kwargs) -> dict:
        """
        Route to appropriate tier based on complexity.
        """
        # Estimate complexity
        tier = self.estimate_complexity(prompt)
        
        # Track routing decision
        self.stats['routing'][tier] = self.stats['routing'].get(tier, 0) + 1
        
        print(f"📊 Routing to {tier} tier")
        
        # Set connections for this tier
        self.connections = self.tiers[tier]
        
        # Use parent failover logic
        return super().complete(prompt, **kwargs)

# Test intelligent routing
router = IntelligentRouter()

test_cases = [
    ("What is WTI?", 'economy'),  # Simple definition
    ("Compare WTI and Brent crude pricing dynamics in 2024.", 'standard'),  # Analysis
    ("""Analyze the interplay between OPEC+ production decisions, U.S. shale output, 
    and global demand recovery in shaping oil prices over the next 6 months. 
    Consider geopolitical risks and inventory dynamics.""", 'premium')  # Complex reasoning
]

for prompt, expected_tier in test_cases:
    print(f"\nPrompt: {prompt[:60]}...")
    result = router.complete(prompt, max_tokens=100)
    print(f"✓ Response from {result['connection_used']}\n")

# Show routing statistics
print("\nRouting Distribution:")
for tier, count in router.stats['routing'].items():
    print(f"  {tier}: {count} requests")

# Auto-graded test
assert router.stats['routing']['economy'] > 0, "Should route some requests to economy"
assert router.stats['routing']['premium'] > 0, "Should route some requests to premium"
print("\n✓ Exercise 3.1 passed!")

## Summary

Congratulations! You've learned to compare and select LLM providers. Key takeaways:

1. **Different models excel at different tasks** - Benchmark on your specific use case
2. **Quality vs cost tradeoffs** - The best model isn't always the most capable
3. **Resilience through failover** - Production systems need backup providers
4. **Intelligent routing** - Route based on task complexity to optimize cost
5. **LLM Mesh simplifies switching** - Change providers without code changes

## Next Steps

- **Module 1**: Learn prompt engineering to get better results from any model
- **Module 2**: Build RAG applications with Knowledge Banks
- **Module 4**: Monitor provider performance in production

## Solutions

<details>
<summary>Exercise 1.1 Solution</summary>

```python
def assess_quality(response: str, criteria: str) -> int:
    score = 3
    response_lower = response.lower()
    
    # Check for key terms
    key_terms = ['crack spread', 'refining margin', 'crude', 'product']
    terms_found = sum(1 for term in key_terms if term in response_lower)
    
    # Check for explanation structure
    has_definition = 'spread' in response_lower and ('difference' in response_lower or 'margin' in response_lower)
    has_relevance = 'trader' in response_lower or 'profit' in response_lower
    
    # Scoring
    if terms_found >= 3 and has_definition and has_relevance:
        score = 5
    elif terms_found >= 2 and (has_definition or has_relevance):
        score = 4
    elif terms_found >= 1:
        score = 3
    else:
        score = 2
    
    return score
```
</details>

<details>
<summary>Exercise 3.1 Solution</summary>

```python
def estimate_complexity(self, prompt: str) -> str:
    prompt_lower = prompt.lower()
    prompt_len = len(prompt.split())
    
    # Check for complexity indicators
    complex_keywords = ['analyze', 'compare', 'evaluate', 'interplay', 'dynamics']
    medium_keywords = ['explain', 'describe', 'discuss']
    simple_keywords = ['what is', 'define', 'list']
    
    complex_count = sum(1 for kw in complex_keywords if kw in prompt_lower)
    medium_count = sum(1 for kw in medium_keywords if kw in prompt_lower)
    simple_count = sum(1 for kw in simple_keywords if kw in prompt_lower)
    
    # Decision logic
    if prompt_len > 50 or complex_count > 0:
        return 'premium'
    elif prompt_len > 15 or medium_count > 0:
        return 'standard'
    else:
        return 'economy'
```
</details>

In [ ]:
key_takeaways([
    "Core concept from this notebook demonstrated with working code",
    "Key parameters and their effects on results",
    "When to apply this technique vs alternatives"
])